In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ============================================================
# STEP 1: Define target state
# ============================================================
print("=" * 60)
print("STEP 1: Target amplitudes")
print("=" * 60)

n_qubits = 3
N = 2**n_qubits  # 8 grid points

# Raw function values: x * e^(-x)
f = np.array([x * np.exp(-x) for x in range(N)])

# Normalize so it's a valid quantum state
psi = f / np.linalg.norm(f)

print(f"\nGrid points: x = 0, 1, 2, ..., {N-1}")
print(f"f(x) = x·e^(-x)")
print(f"\nRaw values:        {f}")
print(f"Normalized ψ(x):   {psi}")
print(f"Sum |ψ|² = {np.sum(psi**2):.6f}  (should be 1)")

print(f"""
Target quantum state:
|ψ⟩ = {psi[0]:.4f}|000⟩ + {psi[1]:.4f}|001⟩ + {psi[2]:.4f}|010⟩ + {psi[3]:.4f}|011⟩
    + {psi[4]:.4f}|100⟩ + {psi[5]:.4f}|101⟩ + {psi[6]:.4f}|110⟩ + {psi[7]:.4f}|111⟩

Note: ψ(0) = 0 because f(0) = 0·e^0 = 0. The node at x=0 is a
feature of the 2S orbital that the 1S doesn't have.
""")

STEP 1: Target amplitudes

Grid points: x = 0, 1, 2, ..., 7
f(x) = x·e^(-x)

Raw values:        [0.     0.3679 0.2707 0.1494 0.0733 0.0337 0.0149 0.0064]
Normalized ψ(x):   [0.     0.7546 0.5552 0.3064 0.1503 0.0691 0.0305 0.0131]
Sum |ψ|² = 1.000000  (should be 1)

Target quantum state:
|ψ⟩ = 0.0000|000⟩ + 0.7546|001⟩ + 0.5552|010⟩ + 0.3064|011⟩
    + 0.1503|100⟩ + 0.0691|101⟩ + 0.0305|110⟩ + 0.0131|111⟩

Note: ψ(0) = 0 because f(0) = 0·e^0 = 0. The node at x=0 is a
feature of the 2S orbital that the 1S doesn't have.



In [2]:
# ============================================================
# STEP 2: Reshape into a tensor
# ============================================================
print("=" * 60)
print("STEP 2: Reshape into a 2×2×2 tensor")
print("=" * 60)

# Each amplitude ψ(x) → ψ(x₁, x₂, x₃) where x = 4*x₁ + 2*x₂ + x₃
# x₁ = most significant bit (qubit 1)
# x₂ = middle bit (qubit 2)  
# x₃ = least significant bit (qubit 3)
T = psi.reshape(2, 2, 2)

print(f"\nT[x₁, x₂, x₃] = ψ(4*x₁ + 2*x₂ + x₃)")
print(f"\nT[0,:,:] (qubit 1 = |0⟩):")
print(T[0])
print(f"\nT[1,:,:] (qubit 1 = |1⟩):")
print(T[1])

STEP 2: Reshape into a 2×2×2 tensor

T[x₁, x₂, x₃] = ψ(4*x₁ + 2*x₂ + x₃)

T[0,:,:] (qubit 1 = |0⟩):
[[0.     0.7546]
 [0.5552 0.3064]]

T[1,:,:] (qubit 1 = |1⟩):
[[0.1503 0.0691]
 [0.0305 0.0131]]


In [3]:
# ============================================================
# STEP 3: First SVD — peel off qubit 1
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: First SVD — peel off qubit 1")
print("=" * 60)

# Reshape: group qubit 1 vs (qubit 2, qubit 3)
# Shape: (2) × (2*2) = 2 × 4
M1 = T.reshape(2, 4)

print(f"\nReshape tensor to matrix M1 of shape {M1.shape}:")
print(f"  Rows = qubit 1 values (0 or 1)")
print(f"  Cols = (qubit 2, qubit 3) combinations")
print(f"\nM1 =")
print(M1)

# SVD: M1 = U @ diag(S) @ Vt
U1, S1, Vt1 = np.linalg.svd(M1, full_matrices=False)

print(f"\nSVD: M1 = U₁ · diag(S₁) · V₁ᵀ")
print(f"\nU₁ (shape {U1.shape}):")
print(U1)
print(f"\nSingular values S₁: {S1}")
print(f"\nV₁ᵀ (shape {Vt1.shape}):")
print(Vt1)

# Determine bond dimension: how many singular values are significant?
threshold = 1e-10
chi1 = np.sum(S1 > threshold)
print(f"\n*** Bond dimension χ₁ = {chi1} (significant singular values) ***")
print(f"    This tells us: qubit 1 has limited entanglement with qubits 2,3")

# Extract A1 matrices: A1[x₁] is row x₁ of U1 (a 1×χ vector)
# Absorb singular values into the remainder
A1 = {0: U1[0:1, :chi1], 1: U1[1:2, :chi1]}
remainder1 = np.diag(S1[:chi1]) @ Vt1[:chi1, :]

print(f"\nA₁[0] = {A1[0]}  (selected when qubit 1 = |0⟩)")
print(f"A₁[1] = {A1[1]}  (selected when qubit 1 = |1⟩)")
print(f"\nRemainder (S₁·V₁ᵀ), shape {remainder1.shape}:")
print(remainder1)


STEP 3: First SVD — peel off qubit 1

Reshape tensor to matrix M1 of shape (2, 4):
  Rows = qubit 1 values (0 or 1)
  Cols = (qubit 2, qubit 3) combinations

M1 =
[[0.     0.7546 0.5552 0.3064]
 [0.1503 0.0691 0.0305 0.0131]]

SVD: M1 = U₁ · diag(S₁) · V₁ᵀ

U₁ (shape (2, 2)):
[[-0.997  -0.0768]
 [-0.0768  0.997 ]]

Singular values S₁: [0.9885 0.1511]

V₁ᵀ (shape (2, 4)):
[[-0.0117 -0.7665 -0.5624 -0.31  ]
 [ 0.9917  0.0724 -0.081  -0.0694]]

*** Bond dimension χ₁ = 2 (significant singular values) ***
    This tells us: qubit 1 has limited entanglement with qubits 2,3

A₁[0] = [[-0.997  -0.0768]]  (selected when qubit 1 = |0⟩)
A₁[1] = [[-0.0768  0.997 ]]  (selected when qubit 1 = |1⟩)

Remainder (S₁·V₁ᵀ), shape (2, 4):
[[-0.0115 -0.7577 -0.5559 -0.3065]
 [ 0.1498  0.0109 -0.0122 -0.0105]]


In [5]:
# ============================================================
# STEP 4: Second SVD — peel off qubit 2
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Second SVD — peel off qubit 2")
print("=" * 60)

# Remainder has shape (χ₁, 4). Reshape to (χ₁*2, 2) 
# to separate qubit 2 from qubit 3
M2 = remainder1.reshape(chi1 * 2, 2)

print(f"\nReshape remainder to M2 of shape {M2.shape}:")
print(f"  Rows = (bond index, qubit 2) combinations")
print(f"  Cols = qubit 3 values")
print(f"\nM2 =")
print(M2)

U2, S2, Vt2 = np.linalg.svd(M2, full_matrices=False)

chi2 = np.sum(S2 > threshold)
print(f"\nSingular values S₂: {S2}")
print(f"\n*** Bond dimension χ₂ = {chi2} ***")

print(f"\nU₂ (shape {U2.shape}):")
print(U2)

print(f"\nVt₂ (shape {Vt2.shape}):")
print(Vt2)

# Extract A2 matrices: for each value of qubit 2, extract a χ₁ × χ₂ block
A2 = {}
U2_truncated = U2[:, :chi2]  # truncate to bond dimension
U2_reshaped = U2_truncated.reshape(chi1, 2, chi2)
for x2 in [0, 1]:
    A2[x2] = U2_reshaped[:, x2, :]
    
# A3 absorbs remaining singular values
A3_full = np.diag(S2[:chi2]) @ Vt2[:chi2, :]
A3 = {}
for x3 in [0, 1]:
    A3[x3] = A3_full[:, x3:x3+1]

print(f"\nA₂[0] (shape {A2[0].shape}, selected when qubit 2 = |0⟩):")
print(A2[0])
print(f"\nA₂[1] (shape {A2[1].shape}, selected when qubit 2 = |1⟩):")
print(A2[1])
print(f"\nA₃[0] (shape {A3[0].shape}, selected when qubit 3 = |0⟩):")
print(A3[0])
print(f"\nA₃[1] (shape {A3[1].shape}, selected when qubit 3 = |1⟩):")
print(A3[1])


STEP 4: Second SVD — peel off qubit 2

Reshape remainder to M2 of shape (4, 2):
  Rows = (bond index, qubit 2) combinations
  Cols = qubit 3 values

M2 =
[[-0.0115 -0.7577]
 [-0.5559 -0.3065]
 [ 0.1498  0.0109]
 [-0.0122 -0.0105]]

Singular values S₂: [0.8643 0.503 ]

*** Bond dimension χ₂ = 2 ***

U₂ (shape (4, 2)):
[[ 0.809  -0.5805]
 [ 0.5819  0.77  ]
 [-0.0808 -0.2644]
 [ 0.0168  0.014 ]]

Vt₂ (shape (2, 2)):
[[-0.3994 -0.9168]
 [-0.9168  0.3994]]

A₂[0] (shape (2, 2), selected when qubit 2 = |0⟩):
[[ 0.809  -0.5805]
 [-0.0808 -0.2644]]

A₂[1] (shape (2, 2), selected when qubit 2 = |1⟩):
[[0.5819 0.77  ]
 [0.0168 0.014 ]]

A₃[0] (shape (2, 1), selected when qubit 3 = |0⟩):
[[-0.3452]
 [-0.4611]]

A₃[1] (shape (2, 1), selected when qubit 3 = |1⟩):
[[-0.7924]
 [ 0.2009]]


In [6]:
# ============================================================
# STEP 5: Verify — reconstruct all amplitudes from MPS
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Verify — reconstruct amplitudes from MPS")
print("=" * 60)
print(f"\nFor each basis state |x₁ x₂ x₃⟩:")
print(f"  ψ(x₁,x₂,x₃) = A₁[x₁] · A₂[x₂] · A₃[x₃]")
print(f"                  (1×χ₁)  (χ₁×χ₂) (χ₂×1)  → scalar")
print()

reconstructed = np.zeros(N)
for x1 in [0, 1]:
    for x2 in [0, 1]:
        for x3 in [0, 1]:
            idx = 4*x1 + 2*x2 + x3
            # Matrix chain multiplication
            val = A1[x1] @ A2[x2] @ A3[x3]
            reconstructed[idx] = val.item()
            bits = f"{x1}{x2}{x3}"
            print(f"  |{bits}⟩: A₁[{x1}]·A₂[{x2}]·A₃[{x3}] = "
                  f"{A1[x1][0]}")
            print(f"          @ {A2[x2]}")  
            print(f"          @ {A3[x3].flatten()}")
            print(f"          = {val.item():.6f}  "
                  f"(target: {psi[idx]:.6f})")
            print()

error = np.linalg.norm(reconstructed - psi)
print(f"Reconstruction error: {error:.2e}")


STEP 5: Verify — reconstruct amplitudes from MPS

For each basis state |x₁ x₂ x₃⟩:
  ψ(x₁,x₂,x₃) = A₁[x₁] · A₂[x₂] · A₃[x₃]
                  (1×χ₁)  (χ₁×χ₂) (χ₂×1)  → scalar

  |000⟩: A₁[0]·A₂[0]·A₃[0] = [-0.997  -0.0768]
          @ [[ 0.809  -0.5805]
 [-0.0808 -0.2644]]
          @ [-0.3452 -0.4611]
          = -0.000000  (target: 0.000000)

  |001⟩: A₁[0]·A₂[0]·A₃[1] = [-0.997  -0.0768]
          @ [[ 0.809  -0.5805]
 [-0.0808 -0.2644]]
          @ [-0.7924  0.2009]
          = 0.754601  (target: 0.754601)

  |010⟩: A₁[0]·A₂[1]·A₃[0] = [-0.997  -0.0768]
          @ [[0.5819 0.77  ]
 [0.0168 0.014 ]]
          @ [-0.3452 -0.4611]
          = 0.555205  (target: 0.555205)

  |011⟩: A₁[0]·A₂[1]·A₃[1] = [-0.997  -0.0768]
          @ [[0.5819 0.77  ]
 [0.0168 0.014 ]]
          @ [-0.7924  0.2009]
          = 0.306372  (target: 0.306372)

  |100⟩: A₁[1]·A₂[0]·A₃[0] = [-0.0768  0.997 ]
          @ [[ 0.809  -0.5805]
 [-0.0808 -0.2644]]
          @ [-0.3452 -0.4611]
          = 0.150278  

In [7]:

# ============================================================
# STEP 6: The circuit interpretation
# ============================================================
print("=" * 60)
print("STEP 6: Circuit interpretation")
print("=" * 60)
print(f"""
The MPS tells us the circuit structure:

  Bond dimensions: 1 → χ₁={chi1} → χ₂={chi2} → 1
  
  This means we need:
  - log₂(χ₁) = {int(np.ceil(np.log2(max(chi1,2))))} bond qubit(s) between qubit 1 and 2
  - log₂(χ₂) = {int(np.ceil(np.log2(max(chi2,2))))} bond qubit(s) between qubit 2 and 3

  Circuit structure (read left to right):
  
  qubit 1:   ─── [ U₁ ] ───────────────────────────
                    │
  bond:      ───── ┘── [ U₂ ] ────────────────────
                          │
  qubit 2:   ──────────── ┘──── [ U₂ ] ───────────
                                   │
  bond:      ────────────────── ── ┘── [ U₃ ] ────
                                          │
  qubit 3:   ──────────────────────────── ┘────────

  Each Uₖ is a small unitary (acts on 1 physical qubit + bond qubits).
  The bond qubits must end in |0⟩ (disentangled) at the end.
  
  For our case with χ₁={chi1}, χ₂={chi2}:
  - Total qubits needed: {n_qubits} physical + {int(np.ceil(np.log2(max(max(chi1,chi2),2))))} bond = {n_qubits + int(np.ceil(np.log2(max(max(chi1,chi2),2))))}
  - Circuit depth: O(n × χ²) = O({n_qubits} × {max(chi1,chi2)}²) = O({n_qubits * max(chi1,chi2)**2})
""")

STEP 6: Circuit interpretation

The MPS tells us the circuit structure:

  Bond dimensions: 1 → χ₁=2 → χ₂=2 → 1

  This means we need:
  - log₂(χ₁) = 1 bond qubit(s) between qubit 1 and 2
  - log₂(χ₂) = 1 bond qubit(s) between qubit 2 and 3

  Circuit structure (read left to right):

  qubit 1:   ─── [ U₁ ] ───────────────────────────
                    │
  bond:      ───── ┘── [ U₂ ] ────────────────────
                          │
  qubit 2:   ──────────── ┘──── [ U₂ ] ───────────
                                   │
  bond:      ────────────────── ── ┘── [ U₃ ] ────
                                          │
  qubit 3:   ──────────────────────────── ┘────────

  Each Uₖ is a small unitary (acts on 1 physical qubit + bond qubits).
  The bond qubits must end in |0⟩ (disentangled) at the end.

  For our case with χ₁=2, χ₂=2:
  - Total qubits needed: 3 physical + 1 bond = 4
  - Circuit depth: O(n × χ²) = O(3 × 2²) = O(12)



In [8]:

# ============================================================
# STEP 7: Compare 1S vs 2S bond dimensions
# ============================================================
print("=" * 60)
print("STEP 7: Compare 1S (e^{-x}) vs 2S (x·e^{-x})")
print("=" * 60)

for label, func_vals in [("e^(-x)  [1S]", np.array([np.exp(-x) for x in range(N)])),
                          ("x·e^(-x) [2S]", np.array([x*np.exp(-x) for x in range(N)]))]:
    psi_cmp = func_vals / np.linalg.norm(func_vals) if np.linalg.norm(func_vals) > 0 else func_vals
    M = psi_cmp.reshape(2, 4)
    _, S_cmp, _ = np.linalg.svd(M, full_matrices=False)
    chi = np.sum(S_cmp > 1e-10)
    print(f"\n  f(x) = {label}")
    print(f"  Singular values at first cut: {S_cmp}")
    print(f"  Bond dimension χ₁ = {chi}")

print(f"""
The 1S orbital has χ=1 (product state!) because:
  e^(-(4x₁+2x₂+x₃)) = e^(-4x₁) · e^(-2x₂) · e^(-x₃)
  
This is why encoding e^(-|x|) was easy for you — it factors
into independent single-qubit rotations. No entangling gates needed!

The 2S orbital has χ=2 because multiplying by x = 4x₁+2x₂+x₃
couples the qubits. But χ=2 is still tiny — just one bond qubit.
""")

STEP 7: Compare 1S (e^{-x}) vs 2S (x·e^{-x})

  f(x) = e^(-x)  [1S]
  Singular values at first cut: [1. 0.]
  Bond dimension χ₁ = 1

  f(x) = x·e^(-x) [2S]
  Singular values at first cut: [0.9885 0.1511]
  Bond dimension χ₁ = 2

The 1S orbital has χ=1 (product state!) because:
  e^(-(4x₁+2x₂+x₃)) = e^(-4x₁) · e^(-2x₂) · e^(-x₃)

This is why encoding e^(-|x|) was easy for you — it factors
into independent single-qubit rotations. No entangling gates needed!

The 2S orbital has χ=2 because multiplying by x = 4x₁+2x₂+x₃
couples the qubits. But χ=2 is still tiny — just one bond qubit.



In [9]:
# ============================================================
# STEP 8: Scale up — how bond dimension grows with qubits
# ============================================================
print("=" * 60)
print("STEP 8: Scaling — more qubits, same low bond dimension")
print("=" * 60)

for n in [4, 6, 8, 10, 12]:
    N_pts = 2**n
    x_vals = np.linspace(0, 10, N_pts)  # fixed physical range
    
    # Test both 1S and 2S type functions
    for label, func in [("e^(-x)", np.exp(-x_vals)), 
                         ("x·e^(-x)", x_vals * np.exp(-x_vals))]:
        psi_test = func / np.linalg.norm(func)
        
        # Find max bond dimension across all cuts
        max_chi = 1
        remaining = psi_test.copy()
        for cut in range(n - 1):
            left_dim = 2 * max_chi
            right_dim = remaining.size // left_dim
            M = remaining.reshape(left_dim, right_dim)
            _, S_test, Vt_test = np.linalg.svd(M, full_matrices=False)
            chi = np.sum(S_test > 1e-10)
            max_chi = chi
            remaining = (np.diag(S_test[:chi]) @ Vt_test[:chi, :]).flatten()
        
        print(f"  n={n:2d} qubits ({N_pts:5d} points), f(x)={label:10s}: "
              f"max bond dim χ = {max_chi}")
    print()

print("""
Notice: bond dimension stays SMALL even as qubit count grows.
This is the magic — smooth functions have low entanglement,
so MPS circuits remain shallow regardless of resolution.
""")

STEP 8: Scaling — more qubits, same low bond dimension
  n= 4 qubits (   16 points), f(x)=e^(-x)    : max bond dim χ = 1
  n= 4 qubits (   16 points), f(x)=x·e^(-x)  : max bond dim χ = 2

  n= 6 qubits (   64 points), f(x)=e^(-x)    : max bond dim χ = 1
  n= 6 qubits (   64 points), f(x)=x·e^(-x)  : max bond dim χ = 2

  n= 8 qubits (  256 points), f(x)=e^(-x)    : max bond dim χ = 1
  n= 8 qubits (  256 points), f(x)=x·e^(-x)  : max bond dim χ = 2

  n=10 qubits ( 1024 points), f(x)=e^(-x)    : max bond dim χ = 1
  n=10 qubits ( 1024 points), f(x)=x·e^(-x)  : max bond dim χ = 2

  n=12 qubits ( 4096 points), f(x)=e^(-x)    : max bond dim χ = 1
  n=12 qubits ( 4096 points), f(x)=x·e^(-x)  : max bond dim χ = 2


Notice: bond dimension stays SMALL even as qubit count grows.
This is the magic — smooth functions have low entanglement,
so MPS circuits remain shallow regardless of resolution.

